# EcoMine Observatory — Stage 1
## Visual feasibility check: can we see a mine, across two sensors?

**Scope of this notebook (deliberately narrow).** Pull Sentinel-2 optical and
Sentinel-1 radar for one known site, compute the four core indices, put
everything on one interactive map, and *look at it*.

That is all. No classifier, no impact indicators, no alerting — those are
Stages 2–4. The golden rule from the Implementation Guide: a small thing that
runs beats a big thing that doesn't.

**What a "go" looks like:** the mine workings are distinguishable from the
surrounding desert in *at least two independent sensors*. Not "the map looks
nice" — two sensors agreeing.

**What this notebook does not do:** it produces no accuracy figure, no
footprint polygon, and no statement about any operator. Everything here is a
visual screening aid.

Licence: GPL-3.0 · Author: Seifeldin M.G. Alkhedir · ORCID 0000-0003-0821-2991

---
## 1. Environment

If `ee.Initialize` fails, run `earthengine authenticate` in a terminal first,
and set `EE_PROJECT` to your registered Google Cloud project ID.

In [ ]:
import os
from pathlib import Path

import ee
import geemap


def load_project_id():
    """يقرأ المعرّف من .env أو من متغيّر البيئة — لا يُكتب داخل الدفتر أبدًا.

    Reads the project ID from .env or the environment. Never hardcode it in a
    notebook: notebook JSON is easy to commit by accident, and a personal Cloud
    project ID does not belong in a public repository.
    """
    env = Path.cwd().parent / ".env"
    if not env.exists():
        env = Path.cwd() / ".env"
    if env.exists():
        for line in env.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line.startswith("EE_PROJECT="):
                return line.split("=", 1)[1].strip().strip("\"'")
    return os.environ.get("EE_PROJECT", "")


EE_PROJECT = load_project_id()
if not EE_PROJECT:
    raise SystemExit(
        "لم يُضبط معرّف المشروع. نفّذ: cp .env.example .env ثم حرّره.\n"
        "Project ID not set. Run: cp .env.example .env, then edit it."
    )

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine ready:", ee.String("ok").getInfo())

---
## 2. Site

Two presets. **Start with `ad_duwayhi`** for the go/no-go check — it is an
open-pit operation, so surface workings are unambiguous at Sentinel-2's 10 m.

`mahd_adh_dhahab` is principally an **underground** mine; its surface signature
is the plant, waste rock and tailings area, which is genuinely smaller. A modest
result there is a correct result, not a failed method. Do not use it to decide
whether the project is viable.

> **Coordinates are APPROXIMATE** and are not surveyed. They are good enough to
> centre a map and no more. Before any validation claim or publication, replace
> them with concession boundaries from the Ministry of Industry & Mineral
> Resources or Ma'aden's public disclosures. If the pit is off-centre when the
> map renders, pan to it and read the real coordinates off the map.

In [ ]:
SITES = {
    "ad_duwayhi": {
        "name": "Ad Duwayhi gold mine (open pit)",
        "lat": 22.44, "lon": 41.55, "buffer_km": 6,
        "note": "Open pit + heap leach. Best visual go/no-go target.",
    },
    "mahd_adh_dhahab": {
        "name": "Mahd adh Dhahab gold mine (underground + surface works)",
        "lat": 23.495, "lon": 40.856, "buffer_km": 5,
        "note": "Underground. Expect a modest surface footprint - this is normal.",
    },
}

SITE_KEY = "ad_duwayhi"        # <-- switch here
site = SITES[SITE_KEY]

aoi = ee.Geometry.Point([site["lon"], site["lat"]]).buffer(site["buffer_km"] * 1000)

print(site["name"])
print(site["note"])

---
## 3. Time window

The Implementation Guide asks for an **end-of-dry-season** composite, following
Brief §2b constraint 1. That rule comes from Sahelian and tropical studies where
wet-season vegetation obscures bare ground.

**It does not apply here.** The Arabian Shield is hyper-arid — there is no wet
season to wait out, and forcing a two-month seasonal window would discard most
of the year's usable imagery for no gain. So this notebook uses a **full-year
low-cloud composite** instead, and says so explicitly rather than silently
deviating.

Keep the seasonal rule for the South African and Sahelian sites, where it earns
its place.

In [ ]:
YEAR = 2025
START, END = f"{YEAR}-01-01", f"{YEAR}-12-31"

WINDOW_RULE = "low_cloud_annual"
WINDOW_JUSTIFICATION = (
    "Site is hyper-arid; the end-of-dry-season rule from Sahelian/tropical "
    "literature is not applicable and was deliberately not applied."
)
print(WINDOW_RULE, "->", WINDOW_JUSTIFICATION)

---
## 4. Sentinel-2 — optical, cloud-masked

Cloud, shadow, cirrus and snow are removed using the Scene Classification Layer
(SCL classes 3, 8, 9, 10, 11).

In [ ]:
def mask_s2(img):
    scl = img.select("SCL")
    bad = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10)).Or(scl.eq(11))
    return img.updateMask(bad.Not()).divide(10000).copyProperties(img, ["system:time_start"])

s2_coll = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(aoi)
           .filterDate(START, END)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
           .map(mask_s2))

n_s2 = s2_coll.size().getInfo()
print(f"Sentinel-2 scenes in window: {n_s2}")
if n_s2 == 0:
    print("INSUFFICIENT DATA - no usable optical scenes. Widen the window.")

---
## 5. Core spectral indices

| Index | Formula | Reads |
|---|---|---|
| NDVI | (B8−B4)/(B8+B4) | vegetation — mines are bare |
| NDWI | (B3−B8)/(B3+B8) | open water — tailings ponds |
| MNDWI | (B3−B11)/(B3+B11) | water including turbid ponds |
| BSI | ((B11+B4)−(B8+B2))/((B11+B4)+(B8+B2)) | exposed mineral surface |
| NDMI | (B8−B11)/(B8+B11) | surface moisture |

In a hyper-arid setting **BSI and NDMI carry most of the signal**. NDVI is near
zero across the whole scene — desert and mine alike — so it discriminates poorly
here. Expect that, and do not read it as a problem with the method.

In [ ]:
def add_indices(img):
    ndvi  = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi  = img.normalizedDifference(["B3", "B8"]).rename("NDWI")
    mndwi = img.normalizedDifference(["B3", "B11"]).rename("MNDWI")
    ndmi  = img.normalizedDifference(["B8", "B11"]).rename("NDMI")
    bsi = img.expression(
        "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
        {"SWIR": img.select("B11"), "RED": img.select("B4"),
         "NIR": img.select("B8"),  "BLUE": img.select("B2")}
    ).rename("BSI")
    return img.addBands([ndvi, ndwi, mndwi, ndmi, bsi])

s2 = s2_coll.map(add_indices).median().clip(aoi)
print("Bands available:", s2.bandNames().getInfo())

---
## 6. Sentinel-1 — radar (the multi-sensor non-negotiable)

One orbit pass only — mixing ascending and descending mixes viewing geometry and
produces artefacts that look like real texture. A 50 m focal median suppresses
speckle without needing a GPU.

**Honest note for this site:** the Arabian Shield is near-permanently
cloud-free, so radar is *not* demonstrating cloud penetration here — it is
contributing surface-roughness texture, which is a genuine but weaker second
signal. The cloud argument gets its real demonstration at the South African
sites.

In [ ]:
s1_base = (ee.ImageCollection("COPERNICUS/S1_GRD")
           .filterBounds(aoi)
           .filterDate(START, END)
           .filter(ee.Filter.eq("instrumentMode", "IW"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))

asc  = s1_base.filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING"))
desc = s1_base.filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
n_asc, n_desc = asc.size().getInfo(), desc.size().getInfo()

s1_coll   = asc if n_asc >= n_desc else desc
pass_used = "ASCENDING" if n_asc >= n_desc else "DESCENDING"

speckle = ee.Kernel.circle(radius=50, units="meters")
vv = s1_coll.select("VV").median().focal_median(kernel=speckle).rename("VV")
vh = s1_coll.select("VH").median().focal_median(kernel=speckle).rename("VH")
s1 = ee.Image.cat([vv, vh]).clip(aoi)

print(f"Sentinel-1: {max(n_asc, n_desc)} scenes, {pass_used} pass "
      f"(asc={n_asc}, desc={n_desc})")

---
## 7. The map

Layers load hidden except true colour and BSI — toggle them in the layer control
at top right and compare.

In [ ]:
m = geemap.Map(center=[site["lat"], site["lon"]], zoom=13)
m.add_basemap("SATELLITE")

m.addLayer(s2, {"bands": ["B4","B3","B2"], "min": 0, "max": 0.3}, "S2 true colour")
m.addLayer(s2, {"bands": ["B12","B8","B4"], "min": 0, "max": 0.4}, "S2 SWIR composite", False)

m.addLayer(s2.select("BSI"),  {"min": -0.3, "max": 0.4,
           "palette": ["01665e","f6e8c3","8c510a"]}, "BSI (bare surface)")
m.addLayer(s2.select("NDVI"), {"min": -0.2, "max": 0.6,
           "palette": ["8c510a","f6e8c3","01665e"]}, "NDVI", False)
m.addLayer(s2.select("NDMI"), {"min": -0.5, "max": 0.5,
           "palette": ["a6611a","f5f5f5","018571"]}, "NDMI", False)
m.addLayer(s2.select("MNDWI"),{"min": -0.5, "max": 0.5,
           "palette": ["8c510a","ffffff","2166ac"]}, "MNDWI (ponds)", False)

m.addLayer(s1.select("VV"), {"min": -25, "max": 0},  "S1 VV (dB)", False)
m.addLayer(s1.select("VH"), {"min": -30, "max": -5}, "S1 VH (dB)", False)

m.addLayer(ee.Image().paint(ee.FeatureCollection([ee.Feature(aoi)]), 0, 2),
           {"palette": ["ffff00"]}, "AOI")
m

---
## 8. The visual check — what to actually look at

Work through these in order. Write down the answers; they become the Stage 1
record.

1. **True colour** — can you find the workings at all? Pit, waste rock, plant,
   any ponds.
2. **BSI** — do the workings sit at markedly higher BSI than the surrounding
   desert? This is the primary discriminator in an arid setting.
3. **NDMI** — is there a moisture contrast at processing areas or ponds?
4. **MNDWI** — are ponds or leach pads picked out as water?
5. **S1 VV** — is there a backscatter texture change over the same area,
   *independently* of the optical bands?

**GO** = the workings are distinguishable in at least two independent sensors
(e.g. BSI *and* VV).

**NOT-GO** = nothing separates from background in any layer. Before concluding
this, check that the AOI is actually centred on the workings — an off-centre
AOI is a far more common cause than a failed method.

### Then be sceptical of your own result

Everything you just identified as "mine" also matches natural bare rock, wadi
beds, sabkha, quarries, roads and construction. A visual match is a hypothesis,
not a detection. It becomes a measurement only after hand-labelled training data
and a confusion matrix — Stage 3. Nothing from this notebook should be quoted
with an accuracy figure, and no statement about any operator follows from it.